# Lab: Coding a Logistic Regression Workflow with scikit-learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/lab-lectures/04-03-lab.ipynb)

## Lab goals

This lab prepares you for Problem Set 3. The lecture notes explain logistic regression
and classification metrics; this notebook explains how to carry out the workflow in
Python with `scikit-learn`.

By the end of the lab, you should be able to:

- prepare a 0/1 target and a numeric feature table;
- split observations into training and test sets reproducibly;
- fit `sklearn.linear_model.LogisticRegression`;
- calculate class-1 probabilities;
- inspect coefficient signs and compare probabilities for profiles that differ by one feature unit;
- create a confusion matrix and compute accuracy, precision, and recall;
- compare a model with a majority-class baseline;
- compare thresholds with a simple test-set payoff calculation;
- measure the operational volume created by a threshold; and
- transform new business profiles so their columns match the training features.

We practice with customer churn data. Problem Set 3 uses the same coding workflow on
credit-card default data, so the graded dataset and business decision remain different.

## How to use this notebook

Open the lab using the course Google Colab link and work from top to bottom. Read each explanation before running the code below it. The **Check** notes describe what you should verify. Short exercises ask you to explain or modify the workflow without making the supplied notebook depend on hidden state. **Runtime > Restart session and run all** should finish without an error.

You do not submit this lab. If you want your changes to persist after you close Colab, select **File > Save a copy in Drive**; otherwise, saving a copy is optional.

Earlier labs introduced DataFrames, grouping, categorical variables, dummy variables,
and column alignment with `reindex`. This lab briefly reminds you of those methods and
spends more time on the new classification workflow.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")

Package roles:

- `pandas` manages tabular data, `numpy` supplies numerical operations, and
  `matplotlib` makes plots.
- `LogisticRegression` is the estimator we will fit.
- `train_test_split` creates reproducible training and test subsets.
- the four imported metric functions evaluate class predictions.

## 1. Load and check the CSV

The instructor-provided `churn_data_source()` helper first searches the current directory and its parent directories for a local course copy of `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`. If it cannot find one—as in a fresh Colab runtime—it loads the CSV from an IBM GitHub URL pinned to a specific commit. You do not need to upload the CSV or mount Google Drive.

`shape`, `head()`, and `columns` are quick checks that the intended file and schema were loaded.

In [ ]:
TELCO_FILE_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
IBM_TELCO_DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "d5371f5d83a446ad5673cbcca3b814b926491f8a/data/Telco-Customer-Churn.csv"
)


def churn_data_source():
    """Return the local course CSV when available, otherwise the pinned IBM URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / TELCO_FILE_NAME
        if local_path.is_file():
            return local_path

    return IBM_TELCO_DATA_URL


data_source = churn_data_source()
churn_raw = pd.read_csv(data_source)

source_location = (
    "local course repository" if isinstance(data_source, Path) else "pinned IBM GitHub source"
)
print(
    f"Loaded {churn_raw.shape[0]:,} rows and {churn_raw.shape[1]} columns "
    f"from the {source_location}."
)
churn_raw.head()

In [ ]:
required_columns = {
    "customerID",
    "tenure",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
}

assert required_columns.issubset(churn_raw.columns)
print("Column check passed.")

**Check:** The raw data should contain 7,043 customers and 21 columns.

## 2. Create the binary target and clean a numeric column

The source target is text. The equality comparison returns `True` for `Yes` and
`False` for `No`; `.astype(int)` converts those values to 1 and 0.

In [ ]:
churn = churn_raw.copy()
churn["churn"] = (churn["Churn"] == "Yes").astype(int)

churn[["Churn", "churn"]].head()

`TotalCharges` is stored as text. `pd.to_numeric(..., errors="coerce")` converts
numeric-looking text and replaces nonnumeric blanks with missing values. The missing
count tells us how many rows need attention.

In [ ]:
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")
churn["TotalCharges"].isna().sum()

For this lab, remove those few rows with `.dropna(subset=...)`. Then divide total
charges by 1,000 so its coefficient uses a readable unit.

In [ ]:
churn = churn.dropna(subset=["TotalCharges"]).copy()
churn["total_charges_1000"] = churn["TotalCharges"] / 1_000

assert set(churn["churn"].unique()) == {0, 1}
assert churn["TotalCharges"].notna().all()

churn[["tenure", "MonthlyCharges", "TotalCharges", "total_charges_1000", "churn"]].head()

### Exercise 2.1 — explain the target

Explain to a partner why `predict_proba()` should later use the probability for class
1, not class 0. State what each class means in this example.

## 3. Inspect class balance and a business grouping

`value_counts()` reports counts. With `normalize=True`, it reports shares instead.

In [ ]:
class_summary = pd.DataFrame(
    {
        "number_of_customers": churn["churn"].value_counts(),
        "share_of_customers": churn["churn"].value_counts(normalize=True),
    }
).rename(index={0: "No churn", 1: "Churn"})

class_summary.round(3)

Most customers did not churn. This is why a high accuracy can be misleading unless
we compare it with a simple baseline and inspect other metrics.

The next method chain groups customers by contract type. The named aggregation
computes the group size and the mean of the 0/1 target; the mean is the churn rate.

In [ ]:
contract_summary = (
    churn.groupby("Contract")
    .agg(
        number_of_customers=("churn", "size"),
        churn_rate=("churn", "mean"),
    )
    .sort_values("churn_rate", ascending=False)
)

contract_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

contract_summary["churn_rate"].plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_xlabel("Contract type")
ax.set_ylabel("Churn rate")
ax.set_title("Month-to-month customers churn more often")
plt.xticks(rotation=0)

plt.show()

**Check:** The churn class is the minority, and month-to-month contracts have the
highest observed churn rate.

## 4. Choose features and create dummy variables

Use three numeric and three categorical business features. The target is deliberately
not included in this list.

In [ ]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "total_charges_1000",
]

categorical_features = [
    "Contract",
    "InternetService",
    "PaymentMethod",
]

model_features = numeric_features + categorical_features

The order supplied to `pd.Categorical` determines which category is first.
`pd.get_dummies(..., drop_first=True)` omits that first category as the baseline.
This repeats a technique from the multivariate-regression lab.

In [ ]:
contract_order = ["Month-to-month", "One year", "Two year"]
internet_order = ["DSL", "Fiber optic", "No"]
payment_order = [
    "Bank transfer (automatic)",
    "Credit card (automatic)",
    "Electronic check",
    "Mailed check",
]

churn["Contract"] = pd.Categorical(churn["Contract"], categories=contract_order)
churn["InternetService"] = pd.Categorical(
    churn["InternetService"],
    categories=internet_order,
)
churn["PaymentMethod"] = pd.Categorical(
    churn["PaymentMethod"],
    categories=payment_order,
)

`scikit-learn` requires a numeric feature matrix. `dtype=int` makes each dummy column
contain 0 and 1 rather than Boolean values.

In [ ]:
X = pd.get_dummies(churn[model_features], drop_first=True, dtype=int)
y = churn["churn"]

X.head()

In [ ]:
assert len(X) == len(y)
assert "churn" not in X.columns
assert X.select_dtypes(exclude="number").shape[1] == 0

print("Feature-table checks passed.")
print("Feature columns:", X.columns.tolist())

**Check:** The omitted categories are month-to-month contract, DSL service, and
automatic bank transfer.

## 5. Split into training and test sets

`train_test_split` returns four objects in matching pairs:

- `X_train` and `y_train` are used to estimate the model;
- `X_test` and `y_test` are held out for this exercise's evaluation.

The arguments have specific roles:

- `test_size=0.30` sends 30% of observations to the test set; and
- `random_state=656` makes the random split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=656,
)

print("Training shape:", X_train.shape)
print("Test shape:    ", X_test.shape)
print(f"Training churn rate: {y_train.mean():.3f}")
print(f"Test churn rate:     {y_test.mean():.3f}")

In [ ]:
assert X_train.columns.tolist() == X_test.columns.tolist()
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}
print("Split checks passed.")

### Exercise 5.1 — reason about the split

What would go wrong if we fitted the model on `X_test` and then described its
performance on the same rows? What does `random_state=656` make reproducible?

## 6. Fit a logistic regression model

The `scikit-learn` estimator workflow has two stages:

1. create an estimator object with chosen settings;
2. call `.fit(X_train, y_train)` to estimate it from the training rows.

`solver="lbfgs"` names the numerical optimizer, and `max_iter=1000` gives it enough
iterations. We otherwise use the estimator defaults because this exercise does not
provide a reason to change them. The estimator includes an intercept automatically;
Module 6 studies regularization choices explicitly.

**A note on the intercept:** In the previous regression labs, we used `statsmodels` and added an intercept column explicitly with `sm.add_constant()`. Here we use scikit-learn's `LogisticRegression`, which includes an intercept by default, so we do **not** add a constant column to `X`.

In [ ]:
logistic_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
)
_ = logistic_model.fit(X_train, y_train)

assert np.array_equal(logistic_model.classes_, [0, 1])
print("Fitted classes:", logistic_model.classes_)
print("Number of fitted feature coefficients:", logistic_model.coef_.shape[1])

**Check:** The classes should be `[0, 1]`, and there should be one coefficient for
every column of `X_train`.

## 7. Predict class-1 probabilities

`.predict_proba(X_test)` returns one row per test customer and one column per fitted
class. Because `logistic_model.classes_` is `[0, 1]`, column 1 is the predicted
probability of churn. The slice `[:, 1]` keeps every row from that column.

In [ ]:
probability_matrix = logistic_model.predict_proba(X_test)
test_probabilities = probability_matrix[:, 1]

print("Probability-matrix shape:", probability_matrix.shape)
test_probabilities[:10].round(3)

In [ ]:
assert len(test_probabilities) == len(y_test)
assert np.all((test_probabilities >= 0) & (test_probabilities <= 1))
assert np.allclose(probability_matrix.sum(axis=1), 1)
print("Probability checks passed.")

## 8. Inspect coefficients and compare predicted probabilities

`coef_` is a two-dimensional array because a classifier can support multiple classes.
For this binary model, `coef_[0]` is the single row of feature coefficients.
`intercept_[0]` retrieves the intercept.

The sign of a feature coefficient indicates the direction in which the trained model's
predicted probability moves when that feature increases and the other included
features are fixed. Its numerical value is not a probability-point change. To find
the probability change for a particular profile, create two otherwise identical
feature rows, increase one feature by one unit, and call `predict_proba()` on both.

In [ ]:
coef_table = pd.DataFrame(
    {
        "feature": X_train.columns,
        "coefficient": logistic_model.coef_[0],
    }
)

intercept_row = pd.DataFrame(
    {
        "feature": ["intercept"],
        "coefficient": [logistic_model.intercept_[0]],
    }
)

coef_table = pd.concat([intercept_row, coef_table], ignore_index=True)

display(coef_table.round(3))

reference_X = X_test.iloc[[0]].copy()
higher_monthly_charge_X = reference_X.copy()
higher_monthly_charge_X["MonthlyCharges"] += 1

probability_check_X = pd.concat(
    [reference_X, higher_monthly_charge_X], ignore_index=True
)
fixed_columns = probability_check_X.columns.drop("MonthlyCharges")
assert probability_check_X[fixed_columns].nunique(dropna=False).eq(1).all()
assert np.isclose(
    probability_check_X.loc[1, "MonthlyCharges"]
    - probability_check_X.loc[0, "MonthlyCharges"],
    1,
)

probability_check_values = logistic_model.predict_proba(probability_check_X)[:, 1]
monthly_charge_coefficient = coef_table.loc[
    coef_table["feature"] == "MonthlyCharges", "coefficient"
].iloc[0]
probability_change = probability_check_values[1] - probability_check_values[0]
probability_change_percentage_points = 100 * probability_change

probability_check = pd.DataFrame(
    {
        "profile": [
            "reference test customer",
            "same customer with MonthlyCharges $1 higher",
        ],
        "MonthlyCharges": probability_check_X["MonthlyCharges"],
        "predicted_churn_probability": probability_check_values,
        "change_from_reference_percentage_points": [
            0.0, probability_change_percentage_points
        ],
    }
)

print(f"MonthlyCharges coefficient: {monthly_charge_coefficient:.4f}")
print(f"Change in predicted probability: {probability_change:.4f}")
print(f"Change in percentage points: {probability_change_percentage_points:.3f}")
print(
    "Are the coefficient and probability change equal?",
    np.isclose(monthly_charge_coefficient, probability_change),
)
probability_check.round(4)

These paired predictions isolate how the trained model's predicted probability changes
for this reference customer when `MonthlyCharges` increases by $1. The other feature
values are identical in the two rows. The coefficient sign and probability-change
direction agree, but the coefficient value and probability change are not the same.
The probability change can also differ for a customer with different starting values.

### Exercise 8.1 — verify a one-unit probability change

Use the printed coefficient and probability-comparison table. Report the two predicted
probabilities and their difference in percentage points. Confirm that a $1 increase in
`MonthlyCharges`, holding the other included features fixed, does not change the
predicted churn probability by the coefficient amount. Describe this as a model
comparison rather than a causal effect.

## 9. Classify at threshold 0.50

A probability becomes an action only after a threshold is chosen. The comparison
returns `True` when a probability is at least 0.50; `.astype(int)` converts the result
to predicted classes 1 and 0.

In [ ]:
threshold = 0.50
test_predictions_50 = (test_probabilities >= threshold).astype(int)

test_predictions_50[:10]

With `labels=[0, 1]`, `confusion_matrix(...).ravel()` returns counts in the fixed order
TN, FP, FN, TP. Naming them on the left makes the order explicit.

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions_50,
    labels=[0, 1],
).ravel()

confusion_50 = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp],
    ],
    index=["Actual no churn", "Actual churn"],
    columns=["Predicted no churn", "Predicted churn"],
)

confusion_50

The metric functions compare actual classes with predicted classes.
`zero_division=0` tells precision to return 0 if a rule predicts no positive cases,
instead of producing an undefined-value warning.

In [ ]:
accuracy = accuracy_score(y_test, test_predictions_50)
precision = precision_score(y_test, test_predictions_50, zero_division=0)
recall = recall_score(y_test, test_predictions_50)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

Read the metrics in business language:

- accuracy: share of all test customers classified correctly;
- precision: among customers predicted to churn, the share who churned; and
- recall: among customers who churned, the share caught by the rule.

## 10. Write a reusable threshold function

We need the same calculations at several thresholds. This function receives actual
classes, probabilities, and one threshold. It returns a dictionary whose keys become
clear table-column names. The body uses the same operations as Section 9 rather than
introducing a new statistical method.

In [ ]:
def classification_metrics(y_true, predicted_probability, threshold):
    predicted_class = (predicted_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted_class, labels=[0, 1]).ravel()

    return {
        "threshold": threshold,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "accuracy": accuracy_score(y_true, predicted_class),
        "precision": precision_score(y_true, predicted_class, zero_division=0),
        "recall": recall_score(y_true, predicted_class),
        "share_predicted_churn": predicted_class.mean(),
    }

Call the function once and compare its results with the direct calculations above.

In [ ]:
metrics_50 = classification_metrics(y_test, test_probabilities, threshold=0.50)
pd.DataFrame([metrics_50]).round(3)

In [ ]:
assert metrics_50["true_negative"] == tn
assert metrics_50["false_positive"] == fp
assert metrics_50["false_negative"] == fn
assert metrics_50["true_positive"] == tp
assert np.isclose(metrics_50["accuracy"], accuracy)
print("Reusable-function check passed.")

## 11. Compare with a majority-class baseline

Because no churn is the majority class, a simple baseline predicts 0 for everyone.
An array of zero probabilities passed through threshold 0.50 produces exactly that
rule. Using the same function ensures a fair metric comparison.

In [ ]:
baseline_probabilities = np.zeros(len(y_test))
metrics_baseline = classification_metrics(
    y_test,
    baseline_probabilities,
    threshold=0.50,
)

baseline_comparison = pd.DataFrame(
    [
        {"rule": "majority-class baseline", **metrics_baseline},
        {"rule": "logistic model at threshold 0.50", **metrics_50},
    ]
)

baseline_comparison[
    [
        "rule",
        "accuracy",
        "precision",
        "recall",
        "share_predicted_churn",
        "true_positive",
        "false_negative",
    ]
].round(3)

**Check:** The baseline may look fairly accurate, but it has zero recall and identifies
no customers for retention action. This is why accuracy alone is not enough.

## 12. Compare thresholds

The list comprehension calls `classification_metrics` once for each candidate and
collects the returned dictionaries. `pd.DataFrame` turns them into one row per
threshold.

In [ ]:
threshold_values = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

threshold_table = pd.DataFrame(
    [
        classification_metrics(y_test, test_probabilities, candidate_threshold)
        for candidate_threshold in threshold_values
    ]
)

threshold_table[
    [
        "threshold",
        "share_predicted_churn",
        "accuracy",
        "precision",
        "recall",
        "false_positive",
        "false_negative",
        "true_positive",
        "true_negative",
    ]
].round(3)

Lower thresholds classify more customers as churners, usually increasing recall and
decreasing precision. Higher thresholds do the reverse. No threshold is automatically
best; the choice depends on the action, benefits, costs, and capacity.

## 13. Add a test-set payoff and measure action volume

Suppose the company sends a retention offer to every customer predicted to churn:

- an offer costs $20;
- retaining a customer who would have churned is worth $100;
- a true positive therefore contributes $80;
- a false positive contributes -$20; and
- false negatives and true negatives contribute $0 in this simplified table.

The calculation below uses the realized confusion counts in the test set, so we call
it **test-set total payoff**, not expected payoff or guaranteed future profit.

In [ ]:
offer_cost = 20
saved_margin = 100

threshold_table["test_set_total_payoff"] = (
    threshold_table["true_positive"] * (saved_margin - offer_cost)
    - threshold_table["false_positive"] * offer_cost
)

threshold_table[
    [
        "threshold",
        "true_positive",
        "false_positive",
        "precision",
        "recall",
        "test_set_total_payoff",
    ]
].round(3)

`.idxmax()` returns the row label where payoff is largest; `.loc[...]` retrieves that
entire row. The selected threshold determines both payoff and workload.

In [ ]:
best_row = threshold_table.loc[threshold_table["test_set_total_payoff"].idxmax()]

review_count = int(best_row["true_positive"] + best_row["false_positive"])
review_share = best_row["share_predicted_churn"]

print(f"Selected threshold: {best_row['threshold']:.2f}")
print(f"Test-set total payoff: ${best_row['test_set_total_payoff']:,.0f}")
print(f"Customers receiving an offer: {review_count:,} out of {len(y_test):,}")
print(f"Share receiving an offer: {review_share:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(threshold_table["threshold"], threshold_table["precision"], marker="o", label="Precision")
axes[0].plot(threshold_table["threshold"], threshold_table["recall"], marker="o", label="Recall")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Metric value")
axes[0].set_title("Precision and recall by threshold")
axes[0].legend()

axes[1].plot(
    threshold_table["threshold"],
    threshold_table["test_set_total_payoff"],
    marker="o",
    color="#F58518",
)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Test-set total payoff ($)")
axes[1].set_title("Payoff and action volume depend on the threshold")

plt.show()

For this introductory lab, the same test set is used to compare thresholds and to
describe the selected threshold. This is a teaching simplification. A production
workflow should select a threshold using validation data and evaluate it once on a
separate final test set.

### Exercise 13.1 — change the action economics

Without changing the supplied notebook permanently, predict what would happen to the
preferred threshold if the offer became much more expensive. Then try one larger
`offer_cost`, rebuild the payoff column, and check your prediction.

## 14. Predict new customer profiles

New profiles must receive exactly the same transformations, category definitions,
feature names, and column order used during training.

In [ ]:
new_customers = pd.DataFrame(
    {
        "profile": [
            "New month-to-month fiber customer",
            "Long-tenure two-year DSL customer",
            "Medium-tenure electronic-check customer",
        ],
        "tenure": [2, 60, 18],
        "MonthlyCharges": [85, 55, 75],
        "TotalCharges": [170, 3300, 1350],
        "Contract": ["Month-to-month", "Two year", "One year"],
        "InternetService": ["Fiber optic", "DSL", "Fiber optic"],
        "PaymentMethod": [
            "Electronic check",
            "Credit card (automatic)",
            "Electronic check",
        ],
    }
)

new_customers["total_charges_1000"] = new_customers["TotalCharges"] / 1_000

new_customers["Contract"] = pd.Categorical(
    new_customers["Contract"],
    categories=contract_order,
)
new_customers["InternetService"] = pd.Categorical(
    new_customers["InternetService"],
    categories=internet_order,
)
new_customers["PaymentMethod"] = pd.Categorical(
    new_customers["PaymentMethod"],
    categories=payment_order,
)

new_customers

`pd.get_dummies` creates columns for the declared category levels.
`reindex(columns=X.columns, fill_value=0)` then puts them in the exact training order
and supplies 0 for any absent dummy column. This alignment is essential before
prediction.

In [ ]:
new_X = pd.get_dummies(new_customers[model_features], drop_first=True, dtype=int)
new_X = new_X.reindex(columns=X.columns, fill_value=0)

assert new_X.columns.tolist() == X.columns.tolist()

new_customers["predicted_churn_probability"] = logistic_model.predict_proba(new_X)[:, 1]
new_customers["send_offer"] = (
    new_customers["predicted_churn_probability"] >= best_row["threshold"]
)

new_customers[
    [
        "profile",
        "predicted_churn_probability",
        "send_offer",
    ]
].round(3)

### Exercise 14.1 — make a profile

In a scratch cell, create one more realistic customer. Apply the same unit conversion,
category definitions, dummy encoding, and column alignment before calling
`predict_proba()`.

## 15. Common mistakes and quick diagnoses

| Symptom | Likely cause | Fix |
|---|---|---|
| Data-loading or network error | Neither a local course CSV nor the pinned IBM source could be read | In Colab, check internet access and rerun the cell; locally, confirm that `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` exists in the current directory or one of its parent directories |
| Missing values or model conversion error | `TotalCharges` remained text or blank | Use `pd.to_numeric(..., errors="coerce")`, inspect missing values, then apply the documented row rule |
| Target appears among features | Outcome leakage | Build `X` only from `model_features` and keep `y` separate |
| Model fails to converge | Optimizer needs more iterations or features have problematic scales | Inspect units and use the documented `max_iter` setting before changing the model |
| Probabilities describe no churn | Wrong probability column | Inspect `classes_`; use the column corresponding to class 1 |
| A coefficient is reported as a probability-point change | The coefficient value was confused with a change in predicted probability | Compare predictions for two otherwise identical rows that differ by one feature unit |
| Confusion counts are mislabeled | `.ravel()` order was assumed incorrectly | Use `labels=[0, 1]` and assign `tn, fp, fn, tp` in that order |
| Precision warns or is undefined | No cases were predicted positive | Use `zero_division=0` and explain what the all-negative rule did |
| Baseline looks strong | Accuracy reflects the majority class | Check recall, confusion counts, and the intended action |
| “Best” threshold changes with costs | Threshold is being treated as universal | Reconnect the threshold to the supplied payoff and capacity assumptions |
| Payoff is called guaranteed profit | Test outcomes were mistaken for the future | Call it test-set total payoff and validate on new data |
| New-profile prediction has wrong columns | Encoding or order differs from training | Reuse category orders and `reindex(columns=X.columns, fill_value=0)` |
| Notebook works only out of order | Hidden kernel state | Restart the kernel and run all cells from top to bottom |

## 16. Transfer checklist for Problem Set 3

Before starting the problem set, make sure you can explain these steps:

1. identify and check the 0/1 target;
2. build numeric and dummy-variable features without target leakage;
3. split with the documented test size and random state;
4. fit an estimator on training data only;
5. retrieve the probability for class 1;
6. inspect a coefficient's direction and use paired predictions to verify that its
   value is not a probability-point change;
7. build and label a confusion matrix;
8. compare accuracy, precision, recall, and a majority-class baseline;
9. calculate test-set total payoff and action volume across thresholds;
10. disclose that using one test set for both threshold selection and evaluation is a
    teaching simplification; and
11. transform and align new profiles before prediction.

If you can explain why each step is needed, you are ready for Problem Set 3.

## 17. Tips for Problem Set 3

You've now completed essentially the same classification workflow you will use in Problem Set 3. The main differences are the **dataset, features, business setting, and operating policy**:

| In this lab | In Problem Set 3 |
|---|---|
| predict customer churn | predict credit-card default |
| investigate churn patterns | investigate September repayment status |
| create numeric and categorical features | use the features specified in the problem set |
| create dummy variables | Same |
| split data into training and test sets | Same |
| fit `LogisticRegression` | Same |
| obtain class-1 probabilities with `predict_proba()` | Same |
| inspect coefficients and compare two otherwise identical cases | Same |
| calculate classification outcomes across thresholds | Same |
| calculate payoff across thresholds | Same |
| consider the number/share receiving an action | apply the specified review-capacity constraint |
| choose an operating threshold | choose the best feasible operating threshold |

The main idea you will transfer is **using predicted probabilities and thresholds to turn a logistic regression model into a business decision rule**.